In [ ]:
!pip install protobuf==3.20.3

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from IPython.display import FileLink

# ================================
# 1. CONFIGURARE
# ================================
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
RANDOM_SEED = 42

base_dir = "/kaggle/input/skin-cancer-mnist-ham10000"
metadata_path = os.path.join(base_dir, "HAM10000_metadata.csv")
image_dir_1 = os.path.join(base_dir, "ham10000_images_part_1")
image_dir_2 = os.path.join(base_dir, "ham10000_images_part_2")

# ================================
# 2. PREGĂTIRE DATE
# ================================
df = pd.read_csv(metadata_path)

image_path_dict = {}
for folder in [image_dir_1, image_dir_2]:
    if os.path.exists(folder):
        for x in os.listdir(folder):
            image_path_dict[x.split(".")[0]] = os.path.join(folder, x)

df["path"] = df["image_id"].map(image_path_dict)
df = df.dropna(subset=["path"])
df["dx"] = df["dx"].astype(str)

train_val_df, test_df = train_test_split(
    df, test_size=0.15, random_state=RANDOM_SEED, stratify=df["dx"]
)

train_df, val_df = train_test_split(
    train_val_df, test_size=0.17, random_state=RANDOM_SEED, stratify=train_val_df["dx"]
)

# ================================
# 3. GENERATOARE
# ================================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=40,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.25,
    shear_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode="nearest"
)

val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col="path", y_col="dx",
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", seed=RANDOM_SEED
)

val_gen = val_test_datagen.flow_from_dataframe(
    val_df, x_col="path", y_col="dx",
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

test_gen = val_test_datagen.flow_from_dataframe(
    test_df, x_col="path", y_col="dx",
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

# ================================
# 4. CLASS WEIGHTS
# ================================
y_train = train_gen.classes
class_weights_vals = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights_vals * 1.5))
mel_index = train_gen.class_indices['mel']
class_weights[mel_index] = class_weights[mel_index] * 1.2 
print(f"Pondere Melanom: {class_weights[mel_index]:.2f}")

# ================================
# 5. MODEL
# ================================
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(256, 256, 3))
base_model.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(7, activation="softmax")(x)

model = models.Model(inputs=base_model.input, outputs=output)
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

# --- ETAPA 1: Antrenare Cap (Head) ---
model.compile(optimizer=optimizers.Adam(1e-3), loss=loss_fn, metrics=["accuracy"])

print("\n==== ETAPA 1: Antrenare Cap (SCURTATĂ LA 6 EPOCI) ====\n")
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=6,
    class_weight=class_weights
)

# --- ETAPA 2: Fine Tuning ---
print("\n==== ETAPA 2: Fine Tuning (SLOW & STEADY) ====\n")
base_model.trainable = True
for layer in base_model.layers[:-70]:
    layer.trainable = False

# Păstrăm Learning Rate-ul mic (1e-5) care a funcționat bine
model.compile(optimizer=optimizers.Adam(1e-5), loss=loss_fn, metrics=["accuracy"])

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weights,
    callbacks=[
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1)
    ]
)

# ================================
# 6. GRAFICE
# ================================
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']

plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.axvline(x=len(history1.history['accuracy'])-1, color='green', linestyle='--', label='Start Fine-Tuning')
plt.title('Evoluție Acuratețe')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Validation Loss')
plt.axvline(x=len(history1.history['loss'])-1, color='green', linestyle='--', label='Start Fine-Tuning')
plt.title('Evoluție Loss')
plt.legend()
plt.grid(True)
plt.show()

# ================================
# 7. EVALUARE
# ================================
pred_prob = model.predict(test_gen)
y_pred = np.argmax(pred_prob, axis=1)
y_true = test_gen.classes
labels = list(test_gen.class_indices.keys())

bal_acc = balanced_accuracy_score(y_true, y_pred)
print(f"\n✅ Balanced Accuracy Final: {bal_acc*100:.2f}%")

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100

plt.figure(figsize=(10,8))
sns.heatmap(cm_norm, annot=True, fmt=".1f", xticklabels=labels, yticklabels=labels, cmap="Blues")
plt.title("Confusion Matrix Normalizată (%)")
plt.ylabel("Clasă Reală")
plt.xlabel("Clasă Prezisă")
plt.show()

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=labels))

model.save('model_final_proiect_optim.h5')
display(FileLink(r'model_final_proiect_optim.h5'))